### script for plotting ga data for kmeans paper.

In [ ]:
import sys
sys.path.append("..")

In [ ]:
import numpy as np
import plotly.graph_objects as go

In [ ]:
import wavelet_transform.utils.bubble_data_manager as bdm
from wavelet_transform.utils.preprocessing import add_noise
from wavelet_transform.utils.extensions import fft
from wavelet_transform.utils.save_plots import save_figure

In [ ]:
savefigs_flag = True

In [ ]:
# Set the sonication pulse parameters
pac = 1e6  # the amplitude of the sine wave in Pa
frequency = 1.1 * 1e6  # insonation frequency (Hz)
period = 1 / frequency  # the period of the acoustic wave at the fundamental frequency 
n_cycles = 100  # no. of cycles of the sine wave
parameters = {
    "time_delay": 0 * period,  # time delay, t1
    "pulse_width": 1000 * n_cycles * period,  # pulse width, W = 1 / Q
    "phase_initial": 0 * period,  # tau
    "phase_shift": 0,  # np.pi / 4,  # phi
    "no_harmonics": 1,   # k
    "Rdot_init": 0,  # initial bubble's wall velocity in m/s
}

# Set sampling rate and noise std
sampling_rate = 200e6  # sampling rate to generate the time vector
signal2noise = 20

r_measure = 1e-4  # distance from the centre of bubble which radiated pressure (acoustic emission from bubble) is calculated over time

Rinit = [.5 * 1e-6, 3.4 * 1e-6, 50 * 1e-6]  # initial bubble radius in m

In [ ]:
Font_Size = 24
Tick_Size = 24
Fig_Height = 500
Fig_Width = 1000

In [ ]:
fig = go.Figure()
fig2 = go.Figure()
colors = ["blue", "red", "black"]
linetypes = ["solid", "dash", "dot"]

for i, R0 in enumerate(Rinit):
    data, t_plot = bdm.get_data(
        R0, frequency, pac, sampling_rate, n_cycles=n_cycles, r_measure=r_measure,
        return_time=True, **parameters
    )

    # Add noise
    np.random.seed(64)
    data = add_noise(data, signal2noise)
    np.random.seed(None)

    fig.add_trace(
        go.Scatter(
            x=t_plot * frequency,
            y=data,
            mode="lines",
            name=f"R0 = {R0*1e6:.1f} µm",
            line=dict(color=colors[i], dash=linetypes[i])
        )
    )

    fft_data, f = fft(data, sampling_rate, frequency)
    fft_data = np.abs(fft_data)
    fig2.add_trace(
        go.Scatter(
            x=f,
            y=10 * np.log10(fft_data),
            mode="lines",
            name=f"R0 = {R0*1e6:.1f} µm",
            line=dict(color=colors[i], dash=linetypes[i])
        )
    )

fig.update_layout(
    width=Fig_Width,
    height=Fig_Height,
    font=dict(size=Font_Size),
    plot_bgcolor="white",
    xaxis=dict(tickfont=dict(size=Tick_Size)),
    yaxis=dict(tickfont=dict(size=Tick_Size)),
)

fig.update_xaxes(
    title="t/T [DL]", 
    range=[0, 7],
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig.update_yaxes(
    title="P<sub>rad</sub> / P<sub>0</sub> [DL]",
    range=[-5, 30],
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
# put legends directly on plot
fig.update_layout(legend=dict(x=.7, y=1, font=dict(size=Tick_Size)))

fig2.update_layout(
    width=Fig_Width,
    height=Fig_Height,
    font=dict(size=Font_Size),
    plot_bgcolor="white",
    xaxis=dict(tickfont=dict(size=Tick_Size)),
    yaxis=dict(tickfont=dict(size=Tick_Size)),
)
fig2.update_xaxes(
    title="f / f<sub>0</sub> [DL]",
    range=[0, 10],
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig2.update_yaxes(
    title="Amplitude [dB]",
    mirror=True,
    ticks="outside",
    showline=True,
    linecolor="black",
    gridcolor="lightgrey",
)
fig2.update_layout(legend=dict(x=0.65, y=1, font=dict(size=Tick_Size)))

fig.show()
fig2.show()

if savefigs_flag is True:
    save_figure(fig, "bd_simulation_time")
    save_figure(fig2, "bd_simulation_freq")